# LLM Landscape & Interview Map

> ⏱️ **This is a volatile-layer chapter. Last reviewed: August 2026.**
>
> Everything in this notebook — model names, hardware constants, serving engines, price
> points — has a shelf life measured in months. The *mechanics* live in llm1–llm6 and are
> stable. If this page is more than a year stale, treat the specifics as history and check
> current sources; the decision frameworks below will still hold.

LLM-track MLE interviews are now a distinct loop at AI-first companies and an increasingly
common round everywhere else. This note maps the terrain: what the model tiers are, which
architectural choices matter for serving, how post-training actually works today, and the
glossary you need to talk about any of it fluently.

## Why This Matters

- The prompt / RAG / fine-tune / post-train decision tree — the single most-asked LLM design question
- Open vs closed model tradeoffs, argued with cost and latency numbers rather than vibes
- Which architectural choices (MoE, GQA, RoPE, context length) show up as serving constraints
- What "reasoning model" means mechanically, and when the extra tokens are worth paying for
- Enough glossary fluency to keep up in a 45-minute design discussion

## Model Tiers (August 2026)

Think in **tiers and capabilities**, not brand names. Brands churn; the tier structure has
been stable for two years.

| Tier | What it is | Typical use |
|---|---|---|
| **Frontier closed** | Largest proprietary models behind an API (OpenAI, Anthropic, Google) | Hardest reasoning, agentic work, anything where quality dominates cost |
| **Frontier open-weight** | Downloadable weights at near-frontier quality (Llama, Qwen, DeepSeek, Mistral families) | Self-hosting for privacy, cost at volume, or customization |
| **Small / efficient** | 1B–20B parameter models, often distilled from a larger sibling | High-QPS classification, extraction, on-device, cost-sensitive paths |
| **Reasoning-tuned** | Models post-trained to emit long chains of thought before answering | Math, code, multi-step planning; costs more tokens and latency |
| **Embedding / reranker** | Encoder models producing vectors or relevance scores | Retrieval — see llm4 |

> 💡 **Interview Tip:** Naming a specific model version rarely scores points and dates you
> if you name a stale one. Saying *"I'd start with a small open-weight model for the
> extraction step and reserve the frontier model for the planning step"* scores every time.

## Architecture Trends That Became Serving Constraints

These are the choices that changed between the original 2017 transformer and what you serve
today. Each one exists because of an inference bottleneck — which is why they show up in
system design rounds rather than just research discussions.

| Choice | What it replaced | Why it exists |
|---|---|---|
| **Mixture of Experts (MoE)** | Dense feed-forward layers | Total params grow, *active* params per token stay small → more capacity per FLOP. Complicates serving: all experts must be resident in memory even though few fire |
| **Grouped-Query Attention (GQA)** | Multi-head attention | Fewer KV heads than Q heads → proportionally smaller KV cache |
| **Multi-head Latent Attention (MLA)** | GQA | Compresses KV into a low-rank latent → further cache reduction |
| **RoPE** | Learned / sinusoidal position embeddings | Relative positions, extrapolates further; scaling tricks extend context post-hoc |
| **RMSNorm** | LayerNorm | Cheaper (no mean subtraction), empirically as good |
| **SwiGLU FFN** | ReLU FFN at 4× width | Gated activation; better quality per parameter |
| **Sliding-window / hybrid attention** | Full causal attention every layer | Caps the O(T²) cost on long contexts |

**The through-line:** every one of these trades a little architectural elegance for memory
or bandwidth at inference. That's the shape of the whole field right now — training compute
is no longer the binding constraint, serving cost is.

In [ ]:
# Why MoE changes the serving conversation.
# Dense: every parameter is read for every token.
# MoE:   only the active experts are read, but ALL of them occupy memory.

def moe_vs_dense(total_params_B, n_experts, experts_per_token, bytes_per_param=1):
    """bytes_per_param: 1 = FP8, 2 = bf16."""
    active_frac = experts_per_token / n_experts
    # Rough split: attention is dense, FFN is the part that gets sharded into experts.
    ffn_share = 0.7
    active_params_B = total_params_B * ((1 - ffn_share) + ffn_share * active_frac)
    return {
        "memory_GB": total_params_B * 1e9 * bytes_per_param / 1e9,   # ALL experts resident
        "active_params_B": active_params_B,                            # only these are read
        "compute_saving": 1 - active_params_B / total_params_B,
    }

print(f"{'config':<34} {'mem (GB)':>10} {'active B':>10} {'FLOP saving':>12}")
for name, kw in [
    ("Dense 70B (bf16)",   dict(total_params_B=70,  n_experts=1,  experts_per_token=1, bytes_per_param=2)),
    ("MoE 200B, 8/64 (FP8)", dict(total_params_B=200, n_experts=64, experts_per_token=8, bytes_per_param=1)),
    ("MoE 400B, 8/128 (FP8)", dict(total_params_B=400, n_experts=128, experts_per_token=8, bytes_per_param=1)),
]:
    r = moe_vs_dense(**kw)
    print(f"{name:<34} {r['memory_GB']:>10.0f} {r['active_params_B']:>10.1f} {r['compute_saving']:>11.0%}")

print()
print("Read this as: MoE buys you FLOPs, not memory.")
print("A 400B MoE still needs 400B params resident — you cannot fit it on one GPU")
print("just because only 8 experts fire per token. This is the #1 MoE misconception.")

## The Decision Framework

This part is evergreen — it has survived every model generation so far.

```
Can a well-written prompt on an off-the-shelf model do it?
  YES → ship that. Measure. Stop.
  NO  ↓
Does it fail because the model lacks FACTS it could look up?
  YES → RAG                                          (llm4)
  NO  ↓
Does it fail because the model lacks a SKILL, FORMAT, or STYLE?
  YES → fine-tune (LoRA first, full FT rarely)       (llm2)
  NO  ↓
Does it fail because the task needs multi-step reasoning or tool use?
  YES → reasoning model, or an agent loop            (llm5)
  NO  ↓
Does it fail on cost or latency at your volume?
  YES → distil to a smaller model; quantize; cache   (llm2, llm3)
  NO  → the task may not be LLM-shaped. Reconsider.
```

> 💡 **Interview Tip:** The trap is jumping to fine-tuning. Interviewers are checking whether
> you reach for the expensive, slow-to-iterate option first. *"Fine-tuning teaches a skill;
> RAG supplies facts; distillation buys cost"* is the one-liner that shows you know the
> difference.

## Open vs Closed

| Dimension | Closed API | Open-weight self-hosted |
|---|---|---|
| **Quality ceiling** | Highest at any given moment | Trails by months, often close enough |
| **Cost shape** | Per-token — linear in volume | Fixed GPU cost — cheap at high utilization, terrible at low |
| **Latency** | Network round-trip + provider queue | You control it; you also own the tail |
| **Privacy** | Data leaves your perimeter | Stays on-prem |
| **Customization** | Limited (prompt, sometimes FT endpoint) | Full — fine-tune, quantize, prune, distil |
| **Ops burden** | None | Real: serving stack, GPU capacity, upgrades, on-call |

**The crossover argument.** Closed APIs win until per-token spend exceeds the fully-loaded
cost of GPUs plus the engineers to run them. That crossover is usually higher than people
expect — a self-hosted deployment at 20% utilization is often more expensive than the API it
replaced. Compute your actual break-even before proposing a migration.

In [ ]:
# Break-even: closed API vs self-hosted open-weight model.
# The number that decides it is UTILIZATION, not the sticker price per GPU-hour.
# All constants below are illustrative — plug in your own before quoting this to anyone.

API_PER_1K = 0.012   # $ per 1k output tokens for a mid-tier hosted model

def self_hosted_cost_per_1k(gpu_hourly, n_gpus, tokens_per_sec_per_gpu,
                            utilization, eng_annual):
    hours_year = 24 * 365
    infra_year = gpu_hourly * n_gpus * hours_year + eng_annual
    tokens_year = tokens_per_sec_per_gpu * n_gpus * utilization * 3600 * hours_year
    if tokens_year == 0:
        return float("inf"), 0.0
    return infra_year / (tokens_year / 1000), tokens_year

print(f"{'utilization':>12} {'tokens/yr':>13} {'$/1k self-host':>16} {'cheaper':>12}")
print("-" * 56)
for util in [0.05, 0.15, 0.35, 0.60, 0.85]:
    cost, toks = self_hosted_cost_per_1k(
        gpu_hourly=3.0,               # 8 GPUs on-demand
        n_gpus=8,
        tokens_per_sec_per_gpu=1200,  # aggregate across batched requests
        utilization=util,
        eng_annual=400_000,           # one loaded senior FTE to run the stack
    )
    verdict = "self-host" if cost < API_PER_1K else "API"
    print(f"{util:>11.0%} {toks/1e9:>12.0f}B {cost:>16.4f} {verdict:>12}")

print(f"\nAPI reference: ${API_PER_1K:.4f} per 1k tokens")
print("Crossover sits between 15% and 35% utilization here.")
print()
print("Two terms people leave out, both of which push toward the API:")
print("  1. The engineer. Remove the $400k and the crossover drops below 10%.")
print("  2. Peakiness. Traffic that peaks at 100% may average 15% — and idle GPUs still bill.")

## Scaling Laws: Two Eras

**Training-compute scaling (Chinchilla, 2022).** For a fixed training budget, model size N
and training tokens T should grow together — roughly T ≈ 20N. A 7B model "wants" ~140B
tokens for compute-optimal training.

**Why nobody trains compute-optimal anymore.** Chinchilla optimizes *training* cost. If you
serve the model billions of times, inference cost dominates the lifetime bill — so you
deliberately over-train a *smaller* model far past the Chinchilla point to get a cheaper
model of equal quality. Modern open models are routinely trained at 100–1000+ tokens per
parameter.

**Test-time compute scaling (2024 onward).** The newer axis: for a *fixed* trained model,
quality improves with tokens spent thinking at inference — longer chains of thought, sampling
multiple candidates and selecting, or search over reasoning paths. This is the mechanism
behind reasoning models, and it reframes the cost conversation: capability is now something
you can buy per-request, not only per-training-run.

**Implication for practitioners:** you now have three dials — model size, training tokens,
and inference tokens. A smaller model given more thinking time can beat a larger model
answering immediately, at lower total cost. Route accordingly.

## Glossary

| Term | One-line definition |
|---|---|
| **Tokenization** | Text → integer IDs, usually via BPE. Tokens ≠ characters ≠ words |
| **Context window** | Max tokens attended at once. Long ≠ free: cost and quality both degrade |
| **KV cache** | Cached keys/values for past tokens so decode doesn't recompute them |
| **Prefill** | Processing the prompt — parallel, compute-bound |
| **Decode** | Emitting tokens one at a time — sequential, memory-bandwidth-bound |
| **Prefix caching** | Reusing KV for a shared prompt prefix across requests |
| **PagedAttention** | KV cache in fixed-size pages → no fragmentation, easy sharing |
| **RadixAttention** | Prefix-tree KV reuse; strong fit for agent loops with shared system prompts |
| **Continuous batching** | Admit/evict requests every forward step instead of per fixed batch |
| **Chunked prefill** | Split long prefills so they don't stall in-flight decodes |
| **MoE** | Mixture of Experts — route each token to a few of many FFN experts |
| **GQA / MLA** | Attention variants that shrink the KV cache |
| **RoPE** | Rotary position embedding — encodes relative position by rotation |
| **RMSNorm** | Normalization without mean subtraction; cheaper than LayerNorm |
| **SwiGLU** | Gated FFN activation; standard in modern LLMs |
| **FP8** | 8-bit float; now common for both weights and KV cache on recent GPUs |
| **SFT** | Supervised fine-tuning on instruction–response pairs |
| **DPO** | Direct Preference Optimization — preference learning without a reward model or RL loop |
| **RLVR** | RL from Verifiable Rewards — reward comes from a checker, not a learned model |
| **GRPO** | Group Relative Policy Optimization — critic-free RL; the workhorse of RLVR |
| **LoRA / QLoRA** | Low-rank adapters; QLoRA adds a quantized frozen base |
| **Distillation** | Train a small student to imitate a large teacher |
| **Reasoning tokens** | Chain-of-thought tokens emitted before the answer; billed, often hidden |
| **RAG** | Retrieval-Augmented Generation — fetch context at query time |
| **Grounding** | Tying output to retrievable, checkable sources |
| **Hallucination** | Fluent, confident, wrong |
| **Speculative decoding** | Small draft model proposes; large model verifies in one pass |
| **Structured output** | Schema-constrained generation — the decoder cannot emit invalid tokens |
| **Tool / function calling** | Model emits a structured call the runtime executes |
| **MCP** | Model Context Protocol — a standard interface between models and tools/data |
| **Eval harness** | The automated suite you run before shipping a prompt or model change |

## Common Interview Questions

**Q: Prompt, RAG, fine-tune, or distil?**
Prompt first — zero cost, instant iteration, and usually enough. RAG when the failure is
missing *facts*, especially facts that change. Fine-tune when the failure is a missing
*skill, format, or style* — prompting can't reliably enforce a rigid output contract at
scale. Distil when quality is already fine and the problem is purely cost or latency. The
common mistake is fine-tuning to inject knowledge; that's RAG's job, and fine-tuning does it
badly and expensively.

**Q: What actually changes when you serve an MoE model?**
Compute per token drops because only a few experts fire, but memory does not — every expert
must be resident. So MoE improves throughput-per-FLOP while making the model *harder* to fit,
which often forces multi-GPU serving with expert-parallel sharding. Additional wrinkle:
routing is load-dependent, so latency is less predictable than a dense model of equal
active-parameter count.

**Q: A stakeholder wants to self-host to save money. How do you evaluate that?**
Compute the break-even honestly. Take current API spend, then model self-hosted cost as GPU
hours at *realistic* utilization plus the engineering time to run the stack. Traffic is
usually peaky, so average utilization is far below peak, and idle GPUs bill anyway. Self-hosting
usually wins on privacy, latency control, or customization long before it wins on cost.

**Q: When is a reasoning model the wrong choice?**
When the task is single-step, when latency budget is tight, or when the extra tokens cost more
than the accuracy is worth. Reasoning models spend tokens to think; on extraction,
classification, or lookup you pay for thinking that buys nothing. Route: cheap model for the
easy majority, reasoning model for the hard tail.

**Q: What's the difference between training-compute and test-time-compute scaling?**
Chinchilla-style scaling says how to trade model size against training tokens for a fixed
*training* budget. Test-time scaling says quality also improves with tokens spent at
*inference* on a fixed model. The practical consequence is that capability became a per-request
purchasing decision, so "which model" is now really "which model at what thinking budget."

## Key Takeaways
- Think in tiers (frontier closed / frontier open / small / reasoning / embedding), not brand names — brands churn
- Modern architecture choices (MoE, GQA, MLA, RoPE, RMSNorm, SwiGLU) all exist to relieve *inference* bottlenecks
- MoE buys FLOPs, not memory: all experts stay resident even though few fire per token
- Decision order: prompt → RAG (facts) → fine-tune (skills) → distil (cost). Never fine-tune to add knowledge
- Open vs closed turns on utilization and ops burden, not sticker price; include the engineer in the math
- Two scaling axes now: training compute (Chinchilla, deliberately violated for serving economics) and test-time compute
- This chapter is the volatile layer — re-verify it yearly; llm1–llm6 hold the durable mechanics